# 🔬 Notebook 3: Deep Dive — Cold Starts, Warm Pools, Concurrency & Async

This is the fun one. We **simulate** the execution data plane in pure Python and watch
each technique earn its keep. Every section is a runnable mini-lab with a **bad → best**
progression.

## 🎯 Learning Objectives

1. Feel the difference between a cold start and a warm start, in numbers.
2. Build a **warm pool** with TTL-based garbage collection.
3. Compare concurrency-limiting strategies (none → per-function → per-account).
4. Implement an **async queue with retries + DLQ**.
5. Pick a worker node with **power-of-two choices** to avoid a central bottleneck.
6. Reason about **provisioned concurrency** and **snapshot-based starts** (SnapStart).

## 🛠️ Setup

```bash
cd 06-system-designs/amazon-lambda
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

> 💡 This lab has **no Docker / no cloud dependencies**. Everything is simulated in pure Python
> so you can see the moving parts clearly. Real Lambda uses Firecracker microVMs, S3, DynamoDB,
> and SQS under the hood — we'll point out where those fit as we go.


## 🧊 Part 1: Cold start vs warm start

A cold start is expensive because we do **all** of this from scratch:

1. Download the code `.zip` from S3 — maybe 30 MB.
2. Boot a Firecracker microVM (own kernel, ~125 ms).
3. Start the language runtime (Python/Node/JVM).
4. Run the user's module-level code (imports, constants).
5. Finally call the handler.

A warm start skips steps 1-4 by **re-using a frozen VM** that already did them.

In [ ]:
import time, random

# Fake timings in ms — roughly what real Lambda looks like for a small Python function.
DOWNLOAD_MS   = 120     # S3 (cached on worker) -> local disk
VM_BOOT_MS    = 125     # Firecracker microVM boot
RUNTIME_MS    = 150     # Python interpreter + import user module
HANDLER_MS    = 5       # the actual user code

def cold_start_ms():
    return DOWNLOAD_MS + VM_BOOT_MS + RUNTIME_MS + HANDLER_MS

def warm_start_ms():
    return HANDLER_MS    # everything else already paid for

print(f"Cold start: {cold_start_ms()} ms")
print(f"Warm start: {warm_start_ms()} ms")
print(f"Speedup:    {cold_start_ms()/warm_start_ms():.0f}×")

A ~80× speedup. That's why **every** FaaS trick below is really "get more warm starts".

## 🏊 Part 2: Warm pool (container reuse)

When an invocation finishes, **don't throw the VM away** — freeze it and put it in a pool.
The next invocation for the same function grabs it from the pool in O(1).
After ~10–15 minutes of idleness, the VM is evicted so we don't hog memory forever.

The subtlety that most sketches of this get wrong: **a warm VM can only serve one
invocation at a time.** So a warm pool does *not* eliminate cold starts — it eliminates
cold starts *in steady state*. Every time concurrency climbs above the number of VMs you
already have, someone pays for a boot. That means our simulation has to model **busy
time**, not just arrivals; otherwise the pool looks magically perfect.

In [ ]:
import heapq, random
from collections import defaultdict, deque

# ❌ Bad: always cold start — throw the VM away after each invocation.
def no_reuse(_fn, _now):
    return cold_start_ms()

# ✅ Best: a pool of frozen VMs with TTL eviction — what real Lambda does.
class WarmPool:
    """
    Discrete-event simulation of one worker fleet.

    Key detail: a VM is either BUSY (running an invocation) or IDLE (frozen in the
    pool). Only IDLE VMs can be reused, which is what makes bursts expensive.
    """
    def __init__(self, ttl_s: float = 600.0, duration_s: float = 0.100):
        self.ttl = ttl_s                      # idle TTL before eviction
        self.duration = duration_s            # how long the handler runs
        self.idle = defaultdict(deque)        # fn -> deque[(vm_id, idle_since)], oldest at left
        self.busy: list = []                  # min-heap of (free_at, fn, vm_id)
        self.cold = self.warm = self.evicted = 0
        self.live = self.peak_live = 0        # VMs currently allocated (busy + idle)

    def _reap(self, now):
        """Invocations that finished by `now` release their VM back to the pool."""
        while self.busy and self.busy[0][0] <= now:
            free_at, fn, vm = heapq.heappop(self.busy)
            self.idle[fn].append((vm, free_at))   # idle_since = when it finished

    def _gc(self, fn, now):
        """Evict VMs that have been idle longer than the TTL. Oldest are leftmost."""
        q = self.idle[fn]
        while q and now - q[0][1] > self.ttl:
            q.popleft()
            self.evicted += 1
            self.live -= 1

    def invoke(self, fn, now) -> int:
        self._reap(now)
        self._gc(fn, now)
        q = self.idle[fn]
        if q:
            vm, _ = q.pop()                   # LIFO: reuse the *hottest* VM, let cold ones age out
            self.warm += 1
            latency = warm_start_ms()
        else:
            vm = f"{fn}-vm{self.cold}"
            self.cold += 1
            self.live += 1
            self.peak_live = max(self.peak_live, self.live)
            latency = cold_start_ms()
        # The VM is unavailable until the handler finishes.
        heapq.heappush(self.busy, (now + latency / 1000 + self.duration, fn, vm))
        return latency


# ── Traffic: 60s of steady load, a 5s 20x burst, then 30s of steady load ──────
random.seed(0)
STEADY_QPS, BURST_QPS, N_FUNCS = 50, 1000, 10
arrivals, t = [], 0.0
for rate, secs in [(STEADY_QPS, 60), (BURST_QPS, 5), (STEADY_QPS, 30)]:
    until = t + secs
    while t < until:
        t += random.expovariate(rate)
        arrivals.append((t, f"fn-{random.randint(0, N_FUNCS-1)}", rate))

pool = WarmPool(ttl_s=600, duration_s=0.100)
per_phase = defaultdict(lambda: [0, 0])       # rate -> [cold, total]
latencies = []
for t, fn, rate in arrivals:
    before_cold = pool.cold
    latencies.append(pool.invoke(fn, t))
    per_phase[rate][0] += pool.cold - before_cold
    per_phase[rate][1] += 1

n = len(latencies)
print(f"invocations: {n:,}   cold: {pool.cold}   warm: {pool.warm}")
print(f"overall cold-start ratio: {pool.cold/n:.2%}")
print(f"peak VMs held:            {pool.peak_live}")
print(f"avg latency:  {sum(latencies)/n:6.1f} ms   (vs {cold_start_ms()} ms with no reuse at all)")
print()
for rate in (STEADY_QPS, BURST_QPS):
    cold, total = per_phase[rate]
    print(f"  at {rate:>4} rps:  {cold:>4} cold / {total:>5} invocations = {cold/total:.2%}")

**What to notice**

- Overall cold-start ratio lands around **4%**, and average latency drops ~19× — but the
  headline number hides the real story.
- Split by phase: at 50 rps the pool costs ~**1%** cold starts; during the 20× burst it
  costs ~**7%**. Nearly every cold start in the whole run happens because demand outran
  the VMs we already had.
- `peak VMs held` is the fleet you actually pay for, and the **burst** sets it, not the average.

That is the whole game: *cold starts are a scaling event, not a steady-state cost.*
Which tells you where to aim the remaining fixes — make scaling up cheap (SnapStart,
code cached on the worker) or make it unnecessary (provisioned concurrency).

⚖️ **What the TTL buys and costs.** Try `ttl_s=10`: the pool evicts aggressively, memory
frees up fast, and the cold-start ratio climbs. Try `ttl_s=3600`: almost no cold starts,
but you are renting RAM for VMs nobody is calling. AWS's real ~10-minute value is a
guess at the median gap between invocations — it is a *business* trade, not a technical one.

## 🚦 Part 3: Concurrency limits & throttling

Auto-scaling "to infinity" is a myth. You must cap concurrency per function (and per account)
so that:

- one buggy function can't monopolise a physical server,
- bursts don't exhaust the fleet and break *other* tenants,
- your cost-attack blast radius is bounded.

In [ ]:
import threading
from collections import Counter

# ❌ Bad: no limits. A runaway function can eat the whole fleet.
class NoLimit:
    def start(self, account, fn): return object(), "ok"
    def finish(self, token): pass

# ⚠️ Better: per-function semaphore. Excess sync calls get 429.
class PerFunctionLimit:
    def __init__(self, cap=100):
        self.cap = cap; self.sem = {}
    def start(self, account, fn):
        s = self.sem.setdefault((account, fn), threading.Semaphore(self.cap))
        if not s.acquire(blocking=False):
            return None, "429 throttled (function)"
        return s, "ok"
    def finish(self, token): token.release()

# ✅ Best: nested per-account + per-function limits.
# The account cap stops one tenant from monopolising the fleet via many small functions.
class AccountAndFunctionLimit:
    def __init__(self, account_cap=1000, fn_cap=100):
        self.account_cap = account_cap
        self.fn_cap      = fn_cap
        self.account_sem = {}
        self.fn_sem      = {}
    def start(self, account, fn):
        asem = self.account_sem.setdefault(account, threading.Semaphore(self.account_cap))
        fsem = self.fn_sem.setdefault((account, fn), threading.Semaphore(self.fn_cap))
        if not asem.acquire(blocking=False):
            return None, "429 throttled (account)"
        if not fsem.acquire(blocking=False):
            asem.release()
            return None, "429 throttled (function)"
        return (asem, fsem), "ok"
    def finish(self, token):
        asem, fsem = token
        fsem.release(); asem.release()


# Simulate 10 *concurrent in-flight* sync invocations against a per-function cap of 3.
limiter = AccountAndFunctionLimit(account_cap=5, fn_cap=3)
in_flight, results = [], []
for _ in range(10):
    token, status = limiter.start("acct-1", "resize")
    results.append(status)
    if token: in_flight.append(token)

print("Outcomes:", Counter(results))
print(f"In-flight after burst: {len(in_flight)}  (per-function cap was 3)")

# Finish them all; capacity is freed so the next request succeeds again.
for t in in_flight: limiter.finish(t)
_, status = limiter.start("acct-1", "resize")
print("After draining, next request:", status)

Notice how *some* requests pass and *some* are throttled — not all-or-nothing.
That's the whole point: graceful degradation under pressure, with a clear 429 signal to the client.

## 📬 Part 4: Async invocation — queue + retries + DLQ

Async requests (`X-Invocation-Type: Event`) return **202 Accepted** immediately.
A **poller** reads from an internal queue (SQS/Kafka) and drives the actual work.

Three things we care about:

1. **Retry with exponential backoff** on transient failures (network, out-of-capacity).
2. **Bounded attempts** (usually 3) so we don't loop forever.
3. **Dead-letter queue (DLQ)** for messages that exhaust all attempts, so a human can inspect them.

In [ ]:
from collections import deque
import random

# ❌ Bad: run once, drop on failure. Lossy!
class DropOnFail:
    def __init__(self):
        self.q = deque(); self.lost = 0
    def submit(self, msg): self.q.append(msg)
    def run_once(self, handler):
        while self.q:
            m = self.q.popleft()
            try: handler(m)
            except Exception: self.lost += 1

# ✅ Best: retries with exponential backoff + DLQ.
class AsyncQueue:
    def __init__(self, max_attempts=3, base_delay_s=1.0):
        self.ready: deque = deque()   # [(msg, attempts_so_far)] — deliverable now
        self.delayed: list = []       # [(visible_at, msg, attempts_so_far)] — in backoff
        self.dlq: list = []
        self.done: list = []
        self.max = max_attempts
        self.base = base_delay_s

    def submit(self, msg):
        self.ready.append((msg, 0))

    def _reheat(self, now):
        """Move messages whose visibility timer expired back onto the ready queue.

        This is SQS's `VisibilityTimeout` in one method: a failed message isn't
        deleted, it just becomes invisible until its backoff elapses.
        """
        still_delayed = []
        for visible_at, msg, attempts in self.delayed:
            if visible_at <= now:
                self.ready.append((msg, attempts))
            else:
                still_delayed.append((visible_at, msg, attempts))
        self.delayed = still_delayed

    def _fail(self, msg, attempts, now):
        attempts += 1
        if attempts >= self.max:
            self.dlq.append(msg)                              # give up; a human looks at it
        else:
            delay = self.base * (2 ** (attempts - 1))         # 1s, 2s, 4s, ...
            self.delayed.append((now + delay, msg, attempts))

    def drain(self, handler, clock, batch_size=10):
        """`clock` is an iterator of timestamps — a virtual clock keeps this deterministic."""
        for now in clock:
            self._reheat(now)
            # A real poller pulls a *batch* per tick, not one message.
            for _ in range(min(batch_size, len(self.ready))):
                msg, attempts = self.ready.popleft()
                try:
                    handler(msg)
                    self.done.append(msg)
                except Exception:
                    self._fail(msg, attempts, now)

    def in_flight(self):
        return len(self.ready) + len(self.delayed)


# Demo: the downstream is flaky 70% of the time.
random.seed(0)
attempts_seen: dict = {}
def flaky_handler(m):
    attempts_seen[m] = attempts_seen.get(m, 0) + 1
    if random.random() < 0.7:
        raise RuntimeError("boom")

N_MSGS = 200                       # enough messages that the DLQ rate is not just noise
q = AsyncQueue(max_attempts=3, base_delay_s=1)
for i in range(N_MSGS):
    q.submit(f"msg-{i}")

# Virtual clock: tick every 0.5 s for 30 s. 4s of backoff fits inside that window.
q.drain(flaky_handler, clock=(i * 0.5 for i in range(60)))

print(f"submitted:  {N_MSGS}")
print(f"succeeded:  {len(q.done)}")
print(f"dead-letter:{len(q.dlq):>3} -> {q.dlq[:3]}{'…' if len(q.dlq) > 3 else ''}")
print(f"still in flight: {q.in_flight()}   (should be 0 — everything reached a terminal state)")
assert len(q.done) + len(q.dlq) == N_MSGS, "a message vanished!"

# With p(fail)=0.7 and 3 attempts, theory says 0.7^3 = 34.3% should end in the DLQ.
print(f"\nDLQ rate: observed {len(q.dlq)/N_MSGS:.0%}  vs  theory 0.7^3 = {0.7**3:.0%}")
print(f"attempts per message: {sorted(attempts_seen.values(), reverse=True)[:8]} …")

The DLQ is the safety net: anything truly broken ends up there for on-call to look at,
instead of silently vanishing. The assertion above is the property that matters —
**every message reaches a terminal state**: succeeded, or dead-lettered. Nothing is lost,
nothing loops forever.

⚖️ **The costs nobody mentions:**

- **Retries are at-least-once, so handlers must be idempotent.** The 30% of messages that
  succeeded on attempt 2 may have partially executed on attempt 1. If the handler charges a
  credit card, you have just charged it twice.
- **Backoff hides outages.** With 1s/2s/4s you find out about a broken downstream a full
  7 seconds late, per message. Alert on DLQ *depth*, not on individual failures.
- **A DLQ nobody reads is a data-loss bug with extra steps.** It needs an owner, an alarm,
  and a redrive path.

## 🎯 Part 5: Scheduling — "power of two choices"

At 600 k QPS, a single central scheduler is a bottleneck. We could round-robin across
worker managers, but that creates hot spots when function/VM sizes vary.

A classic trick: **pick 2 random workers, route to the less-loaded of the two**.
That tiny bit of information-gathering flattens the tail *dramatically* — this is a
well-known result in load-balancing theory.

In [ ]:
import random

N_WORKERS = 200

def random_pick(loads):
    """One random worker. Zero information, zero coordination."""
    return random.randrange(len(loads))

def power_of_two(loads):
    """Sample TWO at random, take the less loaded. One extra probe, huge payoff."""
    a, b = random.sample(range(len(loads)), 2)
    return a if loads[a] <= loads[b] else b

def least_loaded(loads):
    """The 'obvious' answer — and the one that doesn't scale: it needs global state."""
    return min(range(len(loads)), key=lambda i: loads[i])

def simulate(pick_fn, n=20_000):
    loads = [0] * N_WORKERS
    for _ in range(n):
        loads[pick_fn(loads)] += 1
    avg = sum(loads) / len(loads)
    return max(loads), min(loads), avg

random.seed(1)
for name, fn in [("random", random_pick),
                 ("power-of-two", power_of_two),
                 ("least-loaded", least_loaded)]:
    mx, mn, avg = simulate(fn)
    print(f"{name:<14} max={mx:<5} min={mn:<5} avg={avg:.1f}   max/avg = {mx/avg:.2f}")

print("\nrandom      : no coordination, but the tail is ~30% above average.")
print("power-of-two: 2 probes per placement gets you within a few % of perfect.")
print("least-loaded: perfect balance — and it requires an up-to-date, global view of")
print("              all 200 workers on every single placement. At 600k QPS that view")
print("              is stale before you read it, and the read itself is the bottleneck.")

Power-of-two-choices lands within a few percent of the *perfectly* balanced answer while
asking only two workers how they're doing. That's the point: it buys you almost all of the
balance for almost none of the coordination.

⚖️ The honest caveat: this model counts every invocation as one identical unit of load and
never lets load *leave* a worker. Real placement has to weigh memory size, remaining lease
time, and whether the worker already has that function's code cached — and a worker that
looks idle may just be about to receive 500 queued invocations. Real Lambda's Placement
Service is power-of-two *plus* affinity to workers that already have your code.

## 🧊→🚀 Part 6: Provisioned concurrency & SnapStart

Two advanced weapons against cold starts, for completeness:

| Technique | Idea | Trade-off |
|---|---|---|
| **Provisioned concurrency** | Keep N microVMs warm 24/7 for a function; never cold-start until you exceed N. | You pay for idle capacity. |
| **SnapStart / snapshot restore** | Boot the VM once, take a memory snapshot, and *restore* instead of booting. JVM cold starts drop from ~5 s → ~200 ms. | Snapshot correctness (RNG seeds, network sockets) is subtle. |

Below, a tiny simulation of snapshot restore.

In [ ]:
# Pretend snapshot restore skips boot + runtime init, but still needs a quick "resume" step.
SNAPSHOT_RESTORE_MS = 50

def snapstart_ms():
    return SNAPSHOT_RESTORE_MS + HANDLER_MS

print(f"Full cold start:    {cold_start_ms()} ms")
print(f"SnapStart restore:  {snapstart_ms()} ms")
print(f"Warm start:         {warm_start_ms()} ms")

## 🧾 Takeaways

- Cold start is the enemy. **Warm pools** + **code cache on workers** handle 95 % of the problem.
- Concurrency limits per function *and* per account keep one tenant from ruining everyone else's day.
- Async = queue + exponential backoff + DLQ. The DLQ is not optional — it's how you find bugs.
- Scheduling bottlenecks vanish with **power-of-two choices** across sharded worker managers.
- For the last mile: provisioned concurrency (pay for warmth) and SnapStart (skip boot).

## 🔭 What we skipped (and where to look)

- Real **Firecracker** internals: [firecracker-microvm.github.io](https://firecracker-microvm.github.io/).
- Networking isolation (ENI, VPC): real Lambda has a separate control plane just for this.
- Billing at millisecond granularity: out of scope here; conceptually a metering sidecar on each worker.
- Lambda@Edge (run close to the user): same core, different placement strategy.